# Solving a Chemistry Problem Using Quantum

Why Use Quantum Computing for Chemistry?

1. Quantum mechanics governs the behavior of molecules, making QC a natural tool for simulating them
2. Classical computers struggle with: (1) Electron interactions in large molecules (2) Exponential complexity of Schrödinger’s equation
3. Accurate quantum state prediction for materials and drug discovery

How AI Enhances Quantum Chemistry?

1. AI models approximate wavefunctions, speeding up calculations
2. Quantum Neural Networks (QNNs) help predict molecular properties
3. AI-driven optimizers improve quantum simulations

### Problem Statement

Compute the ground‑state energy of the hydrogen molecule (H₂) by simulating its electronic structure on a quantum algorithm.

In [12]:
! pip3 install qiskit qiskit-nature qiskit-algorithms pyscf --break-system-packages

In [13]:
import numpy as np
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import FreezeCoreTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_algorithms import VQE
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorEstimator

#### Step 1: Define molecule and RUN the driver

Simulate the hydrogen (H,) molecule using Qiskit's Quantum Chemistry by defining the Hydrogen Molecule,

In [14]:
driver = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto3g")
es_problem = driver.run()  # This creates the problem object correctly

#### Step 2: Map to Qubit space

Converts the molecular electronic Hamiltonian to a Qubit Hamiltonian via `JordanWignerMapper`

In [15]:
mapper = JordanWignerMapper()

# In newer Qiskit Nature, we get the second_q_ops from the problem
hamiltonian = mapper.map(es_problem.second_q_ops()[0])

#### Step 3: Define Variational Ansatz (Parameterized Quantum Circuit)

Approximate the lowest eigenvalue (ground‑state energy), reported in Hartree

In [16]:
# We need to specify the number of qubits based on the Hamiltonian
num_qubits = hamiltonian.num_qubits
ansatz = TwoLocal(num_qubits, rotation_blocks=["ry", "rz"], entanglement_blocks="cz")

/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_72809/3809579996.py:3: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(num_qubits, rotation_blocks=["ry", "rz"], entanglement_blocks="cz")


#### Step 4: Initialize optimizer and the concrete Estimator

In [17]:
optimizer = COBYLA(maxiter=100)
estimator = StatevectorEstimator()

#### Step 5: Set up and run VQE

Solve Using Variational Quantum Eigensolver (VQE)

In [18]:
vqe = VQE(estimator, ansatz, optimizer)
result = vqe.compute_minimum_eigenvalue(hamiltonian)

print(f"Estimated ground state energy: {result.optimal_value:.4f} Hartree")

Estimated ground state energy: -1.7658 Hartree
